# 14 — Robustesse hors-modèle : le réseau a-t-il appris à couvrir, ou juste mémorisé Heston ?

## La critique numéro un du deep hedging

Un réseau de deep hedging est entraîné **dans un simulateur**. On l'a entraîné sous Heston avec des paramètres précis $P_0 = (v_0, \kappa, \theta, \xi, \rho)$. La question que tout intervieweur quant va poser : *le vrai marché n'est pas ton Heston, avec tes paramètres. Ta politique généralise-t-elle, ou s'est-elle sur-ajustée à ton modèle ?*

On répond en trois temps :
1. **Baseline** : entraîner le hedger sous $P_0$, mesurer son CVaR dans le modèle.
2. **Stress hors-modèle** : évaluer *le même réseau, sans réentraînement* sur des marchés à paramètres différents (vol-of-vol plus forte, autre corrélation, régime de vol plus haut), et comparer sa dégradation à celle du hedge classique.
3. **Randomisation des paramètres** (domain randomization) : réentraîner en tirant les paramètres au hasard à chaque épisode, et montrer que le réseau devient robuste.

On garde le problème simple pour isoler l'effet : vendre un call $K=100$, $T=1$, couvrir avec **le sous-jacent seul** sous coûts (le cadre du notebook 10). Le classique est un delta-hedge BS avec bande de non-trading, dont la vol implicite est **recalibrée au vrai prix** dans chaque scénario (un desk réel observe les prix de marché).


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import norm
from scipy.optimize import brentq

torch.manual_seed(0)
S0, K, mu, r, T = 100., 100., 0.05, 0.02, 1.0
n, cost, alpha = 63, 0.01, 0.95; dt = T/n

def heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T):
    out = []
    for u, b in [(0.5, kappa - rho*xi), (-0.5, kappa)]:
        d = np.sqrt((rho*xi*1j*phi - b)**2 - xi**2*(2*u*1j*phi - phi**2))
        g = (b - rho*xi*1j*phi + d)/(b - rho*xi*1j*phi - d)
        C = r*1j*phi*T + (kappa*theta/xi**2)*((b - rho*xi*1j*phi + d)*T - 2*np.log((1-g*np.exp(d*T))/(1-g)))
        D = (b - rho*xi*1j*phi + d)/xi**2 * ((1-np.exp(d*T))/(1-g*np.exp(d*T)))
        out.append(np.exp(C + D*v0 + 1j*phi*np.log(S0)))
    return out
def heston_call(S0, v0, r, kappa, theta, xi, rho, T, K):
    def integ(phi, i):
        f = heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T)[i]
        return (np.exp(-1j*phi*np.log(K))*f/(1j*phi)).real
    P1 = 0.5 + quad(integ, 1e-8, 200, args=(0,), limit=200)[0]/np.pi
    P2 = 0.5 + quad(integ, 1e-8, 200, args=(1,), limit=200)[0]/np.pi
    return S0*P1 - K*np.exp(-r*T)*P2
def bs_price(S,K,tau,r,s): d1=(np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)); return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d1-s*np.sqrt(tau))
def bs_delta(S,K,tau,r,s): return norm.cdf((np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)))
def cvar(p,a=0.95): l=-p; return l[l>=np.quantile(l,a)].mean()

# scenarios de marche
P0 = (0.04, 2.0, 0.04, 0.3, -0.7)          # in-model (entrainement)
scen = {'P0 in-model':                P0,
        'S1 xi=0.6 (vol-of-vol x2)':  (0.04, 2.0, 0.04, 0.6, -0.7),
        'S2 vol 30% (theta=v0=0.09)': (0.09, 2.0, 0.09, 0.3, -0.7),
        'S3 rho=-0.3 (moins skew)':   (0.04, 2.0, 0.04, 0.3, -0.3),
        'S4 crash rho=-0.9 xi=0.5':   (0.04, 2.0, 0.04, 0.5, -0.9)}
prem0 = heston_call(S0, *[P0[0], r, P0[1], P0[2], P0[3], P0[4]], T, K)
print(f"prime in-model = {prem0:.3f}")


## 1. Le classique face au stress (référence)

On simule chaque marché, on price l'option à son vrai prix (fonction caractéristique), on recalibre la vol implicite du classique à ce prix, et on mesure le CVaR du delta-hedge à bande.


In [ ]:
def sim_np(par, m, seed):
    v0, kappa, theta, xi, rho = par; rng = np.random.default_rng(seed)
    S = np.empty((m, n+1)); v = np.empty((m, n+1)); S[:,0] = S0; v[:,0] = v0
    for k in range(n):
        Z1 = rng.standard_normal(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*rng.standard_normal(m)
        vk = np.maximum(v[:,k], 0.)
        v[:,k+1] = np.maximum(v[:,k] + kappa*(theta-vk)*dt + xi*np.sqrt(vk*dt)*Z2, 0.)
        S[:,k+1] = S[:,k]*np.exp((mu - 0.5*vk)*dt + np.sqrt(vk*dt)*Z1)
    return S, v

def band_hedge(S, premium, sig, band=0.13):
    m = S.shape[0]; times = np.linspace(0, T, n+1); cash = np.full(m, premium); pos = np.zeros(m)
    for k in range(n):
        tau = max(T-times[k], 1e-3); tgt = bs_delta(S[:,k], K, tau, r, sig)
        tr = np.where(np.abs(tgt-pos) > band, tgt-pos, 0.); cash -= tr*S[:,k] + cost*np.abs(tr)*S[:,k]; pos += tr; cash *= np.exp(r*dt)
    return cash + pos*S[:,-1] - np.maximum(S[:,-1]-K, 0.)

classic = {}
for name, par in scen.items():
    prem = heston_call(S0, *[par[0], r, par[1], par[2], par[3], par[4]], T, K)
    sig = brentq(lambda s: bs_price(S0, K, T, r, s) - prem, 1e-3, 2.0)
    S, _ = sim_np(par, 60_000, 7)
    classic[name] = (prem, cvar(band_hedge(S, prem, sig)))
    print(f"{name:28s} prime={prem:6.3f}  vol_imp={sig:.3f}  CVaR classique={classic[name][1]:6.3f}")


On retrouve la référence : le classique tient bien sauf quand la **vol-of-vol** double (S1), car c'est du risque de vega pur, non couvrable avec le sous-jacent, qui gonfle mécaniquement.


## 2. Le réseau entraîné sur P0

Architecture du notebook 10 : un MLP qui voit $[\log(S/K), \tau, \text{position}, v]$ et sort la position. On simule Heston **frais à chaque époque** (le marché est exogène, donc on génère la trajectoire puis on couvre dessus), et on utilise la **perte CVaR empirique directe** (la leçon du notebook 13). 

Point subtil et utile : la prime encaissée n'est qu'une **constante** ajoutée au cash, et le CVaR est invariant par translation à une constante près. Elle ne change donc **pas la politique optimale**, seulement le niveau reporté. On peut entraîner avec une prime fixe et, à l'évaluation, utiliser la vraie prime de chaque scénario.


In [ ]:
def heston_paths_t(par, m):
    v0, kappa, theta, xi, rho = par
    S = torch.full((m,), S0); v = torch.full((m,), float(v0)); Ss = [S]; vs = [v]
    for k in range(n):
        Z1 = torch.randn(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*torch.randn(m)
        vk = torch.clamp(v, min=0.)
        v = torch.clamp(v + kappa*(theta-vk)*dt + xi*torch.sqrt(vk*dt)*Z2, min=0.)
        S = S*torch.exp((mu - 0.5*vk)*dt + torch.sqrt(vk*dt)*Z1)
        Ss.append(S); vs.append(v)
    return torch.stack(Ss, 1), torch.stack(vs, 1)

def cvar_torch(L, a=0.95):
    var = torch.quantile(L, a); return L[L >= var].mean()

class HedgeNet(torch.nn.Module):
    def __init__(self, h=32):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(4, h), torch.nn.ReLU(),
                                       torch.nn.Linear(h, h), torch.nn.ReLU(), torch.nn.Linear(h, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

def hedge_cvar(net, S, v, premium):
    m = S.shape[0]; cash = torch.full((m,), premium); pos = torch.zeros(m)
    for k in range(n):
        tau = float(T - k*dt)
        feat = torch.stack([torch.log(S[:,k]/K), torch.full((m,), tau), pos, v[:,k]], dim=1)
        d = net(feat); tr = d - pos
        cash = cash - tr*S[:,k] - cost*torch.abs(tr)*S[:,k]; cash = cash*np.exp(r*dt); pos = d
    pnl = cash + pos*S[:,-1] - torch.clamp(S[:,-1]-K, min=0.0)
    return pnl

def train(sampler, epochs=400, m=20_000, lr=1e-3):
    net = HedgeNet(); opt = torch.optim.Adam(net.parameters(), lr=lr)
    for ep in range(epochs):
        par = sampler()
        S, v = heston_paths_t(par, m)
        loss = cvar_torch(-hedge_cvar(net, S.detach(), v.detach(), prem0))
        opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1) % 100 == 0: print(f"  epoch {ep+1}  CVaR train = {loss.item():.3f}")
    return net

print("entrainement sur P0 ..."); net_p0 = train(lambda: P0)


## 3. Le test hors-modèle

On évalue `net_p0` (jamais réentraîné) sur chaque scénario, avec la vraie prime, et on met à côté le classique.


In [ ]:
def eval_net(net, par, prem, m=60_000, seed=7):
    Snp, vnp = sim_np(par, m, seed)
    with torch.no_grad():
        pnl = hedge_cvar(net, torch.tensor(Snp, dtype=torch.float32),
                         torch.tensor(vnp, dtype=torch.float32), prem).numpy()
    return cvar(pnl)

print(f"{'scenario':28s} {'classique':>10s} {'net P0':>9s}")
res_p0 = {}
for name, par in scen.items():
    prem = classic[name][0]; cn = eval_net(net_p0, par, prem); res_p0[name] = cn
    print(f"{name:28s} {classic[name][1]:10.2f} {cn:9.2f}")


Lecture attendue : dans le modèle (P0), le réseau bat le classique. Hors modèle, on regarde *qui se dégrade le plus*. Le réseau voit $v$ (la variance instantanée), donc il s'adapte au **niveau** de vol (S2), mais il n'a jamais vu de vol-of-vol ou de corrélation différentes : c'est là qu'on s'attend à de la fragilité (S1, S4).


## 4. Le remède : randomisation des paramètres (domain randomization)

Au lieu d'entraîner sur un seul marché, on tire les paramètres au hasard à chaque épisode. Le réseau ne voit jamais deux fois le même marché, donc il ne peut pas se spécialiser sur un régime : il apprend une politique qui marche **en moyenne sur toute une famille de marchés**. On ne lui donne pas les paramètres en entrée (il ne les connaîtrait pas en vrai) ; il doit être robuste à l'aveugle, en s'appuyant sur ce qu'il observe ($S$, $v$).


In [ ]:
rng_dr = np.random.default_rng(0)
def sampler_dr():
    theta = rng_dr.uniform(0.02, 0.09)          # niveau de vol : 14% a 30%
    xi    = rng_dr.uniform(0.2, 0.6)            # vol-of-vol
    rho   = rng_dr.uniform(-0.9, -0.3)          # skew
    return (theta, 2.0, theta, xi, rho)         # v0 = theta (stationnaire)

print("entrainement avec randomisation des parametres ..."); net_dr = train(sampler_dr, epochs=600)


In [ ]:
print(f"{'scenario':28s} {'classique':>10s} {'net P0':>9s} {'net DR':>9s}")
res_dr = {}
for name, par in scen.items():
    prem = classic[name][0]; cd = eval_net(net_dr, par, prem); res_dr[name] = cd
    print(f"{name:28s} {classic[name][1]:10.2f} {res_p0[name]:9.2f} {cd:9.2f}")


In [ ]:
labels = list(scen.keys()); x = np.arange(len(labels)); wd = 0.27
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x-wd, [classic[k][1] for k in labels], wd, label='classique (bande)', color='tab:orange')
ax.bar(x,    [res_p0[k]    for k in labels], wd, label='reseau entraine sur P0', color='tab:red')
ax.bar(x+wd, [res_dr[k]    for k in labels], wd, label='reseau randomise (DR)', color='tab:green')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('CVaR 95%'); ax.set_title("Robustesse hors-modele (plus bas = mieux)")
ax.legend(); fig.tight_layout(); plt.show()


## Ce qu'il faut retenir

- **Un deep hedger entraîné sur un seul jeu de paramètres est spécialisé, pas universel.** Dans le modèle il bat le classique ; hors modèle il peut se dégrader plus vite, surtout sur les dimensions qu'il n'a jamais vues (vol-of-vol, corrélation), parce qu'il n'observe que $S$ et $v$, pas $\xi$ ni $\rho$.
- **La randomisation des paramètres est le remède standard.** En entraînant sur une famille de marchés, le réseau apprend une politique robuste au régime, au prix d'une légère perte d'optimalité dans le modèle nominal. C'est le compromis classique robustesse contre performance de pointe.
- **La bonne façon de juger un hedger, c'est sa dégradation hors-modèle, pas son chiffre in-model.** C'est exactement le raisonnement qu'un desk applique : un modèle est faux par construction, ce qui compte c'est ce qui arrive quand il l'est.
- Détail d'implémentation utile : la prime est une constante et le CVaR est invariant par translation, donc elle ne change pas la politique apprise, seulement le niveau reporté. Ça permet d'entraîner à prime fixe et d'évaluer à la vraie prime.
